지도 라이브러리 Folium
-  Folium repository :https://github.com/python-visualization/folium
-  Folium QuickStart :https://www.google.com/url?q=https%3A%2F%2Fpython-visualization.github.io%2Ffolium%2Flatest%2F

지도 라이브러리 Folium 설치 확인
- colab 에서는 별도 설치 없이 사용 가능

In [1]:
import folium

seoul = folium.Map(location=[37.55, 126.98], zoom_start=12)
seoul

데이터 수입
- 서울 열린데이터 광장에서 수집 검색'시간대별 역별 승하차'
- https://www.google.com/url?q=https%3A%2F%2Fdata.seoul.go.kr%2FdataList%2FOA-12252%2FS%2F1%2FdatasetView.do
- source data의 품질이 높진 않음
- Kaggle 데이터 활용
 - 서울시 역 코드로 지하철역 위치 조회 정보
 - 서울교통공사 지하철역병 다국어 표기 정보
 - 서울교통공사 연도별, 일별, 시간대별, 역별 승하차 인원 정보
 - https://www.google.com/url?q=https%3A%2F%2Fwww.kaggle.com%2Fdatasets%2Fkimjmin%2Fseoul-metro-usage
 - 오른쪽 상단 dataset download

데이터 불러오기
- 2개의 데이터 활용
 - 승하차 정보: seoul-metro-2021.logs.csv
 - 지하철역 정보: seoul-metro-station-info.csv

In [5]:
data = pd.read_csv('/content/seoul-metro-2021.logs.csv')
data

NameError: name 'pd' is not defined

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

승하차 정보: people_in, people_out

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/10_Lecture/00_Python_chunan/09_Lecture/seoul-metro-2021.logs.csv')
data

- 지하철 역 정보

In [ ]:
station_info = pd.read_csv('/content/drive/MyDrive/10_Lecture/00_Python_chunan/09_Lecture/seoul-metro-station-info.csv')
station_info

데이터 가공

승하차 인원 데이터 상세 정보 확인

In [ ]:
data.info()

역 기준으로 승하차 인원 합치기
- 데이터가 시간으로 쪼개져 있어 groupby() 메서드 활용
- sum()를 사용해 station_code 기준으로 그룹화한 다음 값을 합치기

In [ ]:
station_sum = data.groupby('station_code').sum()

station_sum

In [ ]:
station_sum = station_sum.drop('timestamp', axis=1)
station_sum

지하철 역 데이터 상세 정보 확인하기
- 지하철 역 코드 정보만 있어서 어떤 역인지 확인 불가
- 역 코드에 대응하는 역의 정보를 추가
- 지하철 역 정보 확인

In [ ]:
station_info.info()

- 행 추리기
 - 역코드(station.code)
 - 위도(geo.latitude)
 - 경도(geo.longitude)

In [ ]:
# 필요한 행만 추려내기
station_info = station_info[['station.code', 'geo.latitude', 'geo.longitude']]
 # 데이터 확인하기
station_info

승하차 인원 데이터와 색인 맞추기
- 승하차 인원의 index는 역 코드
- set_index()를 이용해서 동일한 색인 맞추기

In [ ]:
station_info = station_info.set_index('station.code')
station_info

승하차 인원 데이터와 지하철역 데이터 합치기
- join(): 색인을 기준으로 셋 2개를 하나로 합치기

In [ ]:
joined_data = station_sum.join(station_info)

joined_data

데이터 시각화
히트맵(heatmap)
- 데이터가 많은 지역을 뜨겁게 표현
- 데이터가 집중된 곳을 쉽게 식별
- Folium 라이브러리 이용
승차인원 시각화
- folium.Map(): 지도 생성
- location 인수: 위도와 경도
 - zoom_start 어느정도 확대할 것인지 설정

In [ ]:
seoul_in = folium.Map(location=[37.55, 126.98], zoom_start=12)

seoul_in

히트맵에 그리기
- add_to(): 지도 객체에 히트맵 추가

In [ ]:
# 히트맵 플러그인 모듈 탑재
from folium.plugins import HeatMap
# 히트맵 플러그인 지도에 추가하기
HeatMap(data = joined_data[['geo.latitude', 'geo.longitude', 'people_in']]).add_to(seoul_in)
# 지도 확인하기
seoul_in

하차인원 시각화

In [ ]:
seoul_out = folium.Map(location=[37.55, 126.98], zoom_start=12)

HeatMap(data = joined_data[['geo.latitude', 'geo.longitude', 'people_out']]).add_to(seoul_out)

seoul_out

승/하차 히트맵 분석
- 2개의 히트맵이 유사
- 출/퇴근 시간을 고려하지 않고 작성하여 유사

데이터 2차 가공 및 시각화  
시간을 고려한 데이터 재가공


In [ ]:
# 데이터 상세 정보 확인
data.info()

- timestamp는 object 문자열
- object -> timestamp 타입으로 변경
- 시간 정보를 추출해 원하는 시간 조건을 넣음
- 출/퇴근용 데이셋 추출
 - 출근 데이터셋은 오전9시 이후
 - 퇴근 데이터셋은 오후5시 이후

출근 데이터 추출

In [ ]:
morning_data = data[pd.to_datetime(data.timestamp).dt.hour < 9]
morning_data

퇴근 데이터 추출

In [ ]:
evening_data = data[pd.to_datetime(data.timestamp).dt.hour>17]
evening_data

그룹화 및 역 정보 합치기
- 역 기준으로 그룹화하여 데이터 더하기

In [ ]:
morning_station_sum = morning_data.groupby('station_code').sum()
evening_station_sum = evening_data.groupby('station_code').sum()

- 역 정보 합치기

In [ ]:
morning_joined_data = morning_station_sum.join(station_info)
evening_joined_data = evening_station_sum.join(station_info)

시각화

출근 시간 승하차 인원 시각화

In [ ]:
# 출근 시간 승차 인원
morning_seoul_in = folium.Map(location=[37.55, 126.98], zoom_start = 12)
HeatMap(data = morning_joined_data[['geo.latitude', 'geo.longitude', 'people_in']]).add_to(morning_seoul_in)
morning_seoul_in

In [ ]:
# 출근 시간 하차 인원
morning_seoul_out = folium.Map(location=[37.55, 126.98], zoom_start = 12)
HeatMap(data = morning_joined_data[['geo.latitude', 'geo.longitude', 'people_out']]).add_to(morning_seoul_out)
morning_seoul_out

퇴근 시간 승하차 인원 시각화

In [ ]:
 # 퇴근 시간 승차 인원
evening_seoul_in = folium.Map(location=[37.55, 126.98], zoom_start = 12)
HeatMap(data = evening_joined_data[['geo.latitude', 'geo.longitude', 'people_in']]).add_to(evening_seoul_in)
evening_seoul_in

In [ ]:
 # 퇴근 시간 하차 인원
Programming_014.ipynb - Colab
Leaflet | © OpenStreetMap contributors
evening_seoul_out = folium.Map(location=[37.55, 126.98], zoom_start = 12)
HeatMap(data = evening_joined_data[['geo.latitude', 'geo.longitude', 'people_out']]).add_to(evening_seoul_out)
evening_seoul_out

마무리
- 출/퇴근 시간에 사람이 몰리는 붉은색 점으로 확인할 수 있었음